# Do Longer Hours Drive More Foot Traffic?
**Negative Binomial Regression · GLM · 106K+ Yelp Businesses**

Yelp uses business attributes including operating hours to rank and recommend businesses to users. This project tested whether operating hours are a reliable predictor of customer foot traffic, and whether they deserve the weight they carry in platform decisions.

**Research questions:**
- Do operating hours meaningfully predict customer check-ins?
- Which factors are actually stronger predictors of foot traffic?
- Does the effect of hours hold after controlling for business type and region?

## 1. Setup & Imports

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

from IPython.display import Image, display

warnings.filterwarnings('ignore')

BUSINESS_JSON_PATH = '/content/drive/MyDrive/yelp/yelp_academic_dataset_business.json'
CHECKIN_JSON_PATH  = '/content/drive/MyDrive/yelp/yelp_academic_dataset_checkin.json'
PLOTS_DIR          = 'plots'
os.makedirs(PLOTS_DIR, exist_ok=True)

for old_file in ['hours_distribution.png','checkins_distribution.png',
    'hours_vs_checkins.png','region_comparison.png','model_fit.png']:
    p = os.path.join(PLOTS_DIR, old_file)
    if os.path.exists(p): os.remove(p)

print('Setup complete.')

## 2. Visual Style & Helpers

In [ ]:
LIME  = '#c8ff00'
PINK  = '#ff69b4'
WHITE = '#ffffff'
BG    = '#0a0a0a'
GRAY  = '#333333'
MGRAY = '#1a1a1a'

plt.rcParams.update({
    'figure.figsize':   (10, 6),
    'figure.facecolor': BG,
    'axes.facecolor':   BG,
    'axes.edgecolor':   GRAY,
    'axes.labelcolor':  WHITE,
    'xtick.color':      WHITE,
    'ytick.color':      WHITE,
    'text.color':       WHITE,
    'axes.titleweight': 'bold',
    'axes.titlesize':   14,
    'axes.labelsize':   11,
})

def apply_style(ax):
    ax.set_facecolor(BG)
    ax.grid(True, axis='y', color=MGRAY, linewidth=0.8)
    ax.grid(False, axis='x')
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)
    ax.spines['left'].set_color(GRAY)
    ax.spines['bottom'].set_color(GRAY)

def save_plot(filename):
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, filename), dpi=300, bbox_inches='tight', facecolor=BG)
    plt.show()
    plt.close()

def calculate_hours(time_range):
    """Convert '9:0-17:30' into total open hours. Handles overnight shifts."""
    if pd.isna(time_range) or str(time_range).strip().lower() in {'none','','nan'}:
        return 0.0
    try:
        open_time, close_time  = str(time_range).split('-')
        oh, om = map(int, open_time.split(':'))
        ch, cm = map(int, close_time.split(':'))
        open_total  = oh * 60 + om
        close_total = ch * 60 + cm
        if close_total < open_total: close_total += 24 * 60
        return (close_total - open_total) / 60.0
    except Exception:
        return 0.0

def map_region(state):
    state  = str(state).strip().upper()
    west   = {'AZ','NV','CA','WA','OR','HI','AK'}
    middle = {'OH','WI','IL','MI','IN','MN','MO','IA','KS','NE','ND','SD'}
    east   = {'PA','NC','SC','NY','NJ','MA','FL','GA','VA','MD','DC','DE','CT','RI','VT','NH','ME','ON','QC'}
    if state in west:   return 'West'
    if state in middle: return 'Middle'
    if state in east:   return 'East'
    return 'Other'

def pseudo_r2(model):
    return 1 - (model.llf / model.llnull)

print('Style and helpers ready.')

## 3. Load Dataset

> **Data:** Yelp Academic Dataset — https://www.yelp.com/dataset
> Place both JSON files in a folder called `yelp` in your Google Drive.

In [ ]:
print('Loading raw Yelp files...')
business_raw = pd.read_json(BUSINESS_JSON_PATH, lines=True)
checkin_raw  = pd.read_json(CHECKIN_JSON_PATH,  lines=True)
print('Business raw shape:', business_raw.shape)
print('Check-in raw shape:', checkin_raw.shape)

## 4. Build Analysis-Ready Dataset

In [ ]:
business = business_raw[['business_id','name','state','stars','review_count','categories']].copy()

# Parse operating hours
hours_rows = []
for _, row in business_raw.iterrows():
    hours = row.get('hours', {})
    if pd.isna(hours) or hours is None: hours = {}
    hours_rows.append({
        'business_id': row['business_id'],
        'monday':    hours.get('Monday'),
        'tuesday':   hours.get('Tuesday'),
        'wednesday': hours.get('Wednesday'),
        'thursday':  hours.get('Thursday'),
        'friday':    hours.get('Friday'),
        'saturday':  hours.get('Saturday'),
        'sunday':    hours.get('Sunday'),
    })
business_hours = pd.DataFrame(hours_rows)

# Count check-in events
checkin = checkin_raw.copy()
checkin['checkins'] = checkin['date'].apply(lambda x: len(str(x).split(',')) if pd.notna(x) else 0)
checkin = checkin[['business_id','checkins']].copy()

In [ ]:
# Compute weekly hours and merge
days = ['monday','tuesday','wednesday','thursday','friday','saturday','sunday']
for day in days:
    business_hours[day] = business_hours[day].apply(calculate_hours)
business_hours['total_hours_per_week'] = business_hours[days].sum(axis=1)
business_hours = business_hours[['business_id','total_hours_per_week']].copy()

checkin_summary = checkin.groupby('business_id', as_index=False)['checkins'].sum()

df = business.merge(business_hours, on='business_id', how='left')
df = df.merge(checkin_summary, on='business_id', how='left')

df['total_hours_per_week'] = df['total_hours_per_week'].fillna(0)
df['checkins']             = df['checkins'].fillna(0)
df['stars']                = pd.to_numeric(df['stars'], errors='coerce').fillna(0)
df['review_count']         = pd.to_numeric(df['review_count'], errors='coerce').fillna(0)
df['categories']           = df['categories'].fillna('').astype(str)
df['is_restaurant']        = df['categories'].str.contains('Restaurant', case=False, na=False).astype(int)
df['region']               = df['state'].apply(map_region)
df = df[(df['total_hours_per_week'] > 0) & (df['checkins'] > 0)].copy()
df['hours_stars']   = df['total_hours_per_week'] * df['stars']
df['hours_reviews'] = df['total_hours_per_week'] * df['review_count']

print('=' * 60)
print('DATASET SUMMARY')
print('=' * 60)
print(f'Businesses analyzed: {len(df):,}')
print(df[['total_hours_per_week','checkins','stars','review_count']].describe())

In [ ]:
# Hours and check-ins distributions
fig, ax = plt.subplots()
sns.histplot(df['total_hours_per_week'], bins=25, color=LIME, edgecolor=BG, linewidth=0.4, ax=ax)
ax.set_title('Most Businesses Operate Within a Similar Weekly Hour Range', pad=12)
ax.set_xlabel('Total Hours Open Per Week')
ax.set_ylabel('Number of Businesses')
apply_style(ax)
save_plot('hours_distribution.png')

fig, ax = plt.subplots()
sns.histplot(np.log1p(df['checkins']), bins=50, color=PINK, edgecolor=BG, linewidth=0.3, ax=ax)
ax.set_title('Most Businesses Receive Low Customer Traffic', pad=12)
ax.set_xlabel('Customer Traffic (scaled for readability)')
ax.set_ylabel('Number of Businesses')
apply_style(ax)
save_plot('checkins_distribution.png')

## 5. Negative Binomial Regression

Negative Binomial GLM used because check-in counts are overdispersed count data.

Three iterative models built progressively:
- **Model 1:** Hours only
- **Model 2:** Hours + business type + region + stars + review count
- **Model 3:** Model 2 + interaction terms

In [ ]:
formula_1 = 'checkins ~ total_hours_per_week'
formula_2 = 'checkins ~ total_hours_per_week + stars + review_count + is_restaurant + C(region)'
formula_3 = 'checkins ~ total_hours_per_week + stars + review_count + is_restaurant + C(region) + hours_stars + hours_reviews'

print('Fitting Model 1 (hours only)...')
model_1 = sm.GLM.from_formula(formula=formula_1, data=df, family=sm.families.NegativeBinomial()).fit()
print('Fitting Model 2 (hours + controls)...')
model_2 = sm.GLM.from_formula(formula=formula_2, data=df, family=sm.families.NegativeBinomial()).fit()
print('Fitting Model 3 (full model with interactions)...')
model_3 = sm.GLM.from_formula(formula=formula_3, data=df, family=sm.families.NegativeBinomial()).fit()

print('\n' + '=' * 70)
print('MODEL COMPARISON')
print('=' * 70)
print(f'Model 1 pseudo R²: {pseudo_r2(model_1):.4f}  (hours only)')
print(f'Model 2 pseudo R²: {pseudo_r2(model_2):.4f}  (+ controls)')
print(f'Model 3 pseudo R²: {pseudo_r2(model_3):.4f}  (+ interactions)')
print(f'\nHours coefficient:')
print(f'  Model 1: {model_1.params["total_hours_per_week"]:.4f}')
print(f'  Model 2: {model_2.params["total_hours_per_week"]:.4f}')
print(f'  Model 3: {model_3.params["total_hours_per_week"]:.4f}')

print('\n--- Model 1 Full Summary ---')
print(model_1.summary())
print('\n--- Model 2 Full Summary ---')
print(model_2.summary())
print('\n--- Model 3 Full Summary ---')
print(model_3.summary())

In [ ]:
# Hours vs check-ins — the key chart
scatter_sample = df.sample(min(20000, len(df)), random_state=42)

fig, ax = plt.subplots()
sns.scatterplot(data=scatter_sample, x='total_hours_per_week', y='checkins',
                alpha=0.2, s=8, color=LIME, edgecolor=None, ax=ax)
sns.regplot(data=scatter_sample, x='total_hours_per_week', y='checkins',
            scatter=False, lowess=True, color=PINK, line_kws={'linewidth': 2}, ax=ax)
ax.set_yscale('log')
ax.set_title('Longer Operating Hours Do Not Strongly Increase Customer Traffic', pad=12)
ax.set_xlabel('Total Hours Open Per Week\n(e.g. 70 hrs = 10 hrs/day, 7 days)')
ax.set_ylabel('Customer Check-ins\n(log scale — each step = 10x more visits)')
apply_style(ax)
save_plot('hours_vs_checkins.png')

# Region comparison
region_summary = df.groupby('region', as_index=False)['checkins'].mean()
region_order   = ['West','Middle','East','Other']
region_summary['region'] = pd.Categorical(region_summary['region'], categories=region_order, ordered=True)
region_summary = region_summary.sort_values('region')

fig, ax = plt.subplots()
sns.barplot(data=region_summary, x='region', y='checkins', color=LIME, ax=ax)
ax.set_title('Customer Traffic Varies More by Region Than by Hours', pad=12)
ax.set_xlabel('Region')
ax.set_ylabel('Average Customer Traffic')
apply_style(ax)
save_plot('region_comparison.png')

In [ ]:
# Actual vs predicted (Model 3)
fit_sample = df.sample(min(15000, len(df)), random_state=42).copy()
fit_sample['predicted_checkins'] = model_3.predict(fit_sample)
max_val = max(fit_sample['checkins'].quantile(0.99), fit_sample['predicted_checkins'].quantile(0.99))

fig, ax = plt.subplots()
sns.scatterplot(data=fit_sample, x='checkins', y='predicted_checkins',
                alpha=0.18, s=10, color=LIME, edgecolor=None, ax=ax)
ax.plot([1, max_val], [1, max_val], linestyle='--', color=GRAY, linewidth=1.5)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_title('Model Captures Overall Trends but Not Exact Outcomes', pad=12)
ax.set_xlabel('Actual Customer Traffic (scaled)')
ax.set_ylabel('Predicted Customer Traffic (scaled)')
apply_style(ax)
save_plot('model_fit.png')

## 6. Key Findings Summary

In [ ]:
print('=' * 70)
print('KEY FINDINGS SUMMARY')
print('=' * 70)
print(f"""
Dataset: {len(df):,} businesses

MODEL FIT (McFadden pseudo R²)
  Model 1 (hours only):     {pseudo_r2(model_1):.4f}
  Model 2 (+ controls):     {pseudo_r2(model_2):.4f}
  Model 3 (+ interactions): {pseudo_r2(model_3):.4f}

KEY FINDING
  Operating hours have a statistically significant but negligible
  effect on customer foot traffic (β=0.0135, p<.001)
  The effect exists but is too small to be meaningful for ranking decisions

  Business type, region, and review count are far stronger predictors
  Adding these signals improved model fit from 0.3% to 12.35%
  Hours alone explain almost none of the variation in foot traffic
""")
print('Region average check-ins:')
print(region_summary.to_string(index=False))

## 7. Preview All Charts

In [ ]:
charts = [
    'hours_vs_checkins.png',
    'region_comparison.png',
    'hours_distribution.png',
    'checkins_distribution.png',
    'model_fit.png'
]
print(f'Saved: {sorted(os.listdir(PLOTS_DIR))}')
for chart in charts:
    path = os.path.join(PLOTS_DIR, chart)
    if os.path.exists(path):
        print(f'--- {chart} ---')
        display(Image(filename=path))